# 时间序列聚类

In [ ]:
import pprint
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
import scipy.spatial.distance as ssd
from sklearn.metrics.pairwise import cosine_distances
from sklearn.metrics import v_measure_score

from reservoir_computing.modules import RC_model
from reservoir_computing.datasets import ClfLoader

np.random.seed(0) # 固定随机种子以确保可重复性

## 配置 RC 模型

In [ ]:
config = {}

# Reservoir（储层）
config['n_internal_units'] = 450        # 储层的大小
config['spectral_radius'] = 0.9         # 储层的最大特征值
config['leak'] = None                   # 储层状态更新中的泄漏量（None 或 1.0 表示无泄漏）
config['connectivity'] = 0.25           # 储层中非零连接的百分比
config['input_scaling'] = 0.1           # 输入权重的缩放
config['noise_level'] = 0.0             # 储层状态更新中的噪声
config['n_drop'] = 5                    # 要丢弃的瞬态状态数
config['bidir'] = True                  # 如果为 True，使用双向储层
config['circle'] = False                # 使用圆形拓扑的储层

# Dimensionality reduction（降维）
config['dimred_method'] ='tenpca'       # 选项：{None（无降维）, 'pca', 'tenpca'}
config['n_dim'] = 75                    # 降维过程后的结果维度数

# MTS representation（多元时间序列表示）
config['mts_rep'] = 'reservoir'         # MTS 表示：{'last', 'mean', 'output', 'reservoir'}
config['w_ridge_embedding'] = 5.0       # 岭回归的正则化参数

# Readout（读出层）
config['readout_type'] = None           # 设置为 None 时，将存储输入表示

pprint.pprint(config)

{'bidir': True,
 'circle': False,
 'connectivity': 0.25,
 'dimred_method': 'tenpca',
 'input_scaling': 0.1,
 'leak': None,
 'mts_rep': 'reservoir',
 'n_dim': 75,
 'n_drop': 5,
 'n_internal_units': 450,
 'noise_level': 0.0,
 'readout_type': None,
 'spectral_radius': 0.9,
 'w_ridge_embedding': 5.0}


## 准备数据

In [3]:
Xtr, Ytr, Xte, Yte = ClfLoader().get_data('Japanese_Vowels')

Loaded Japanese_Vowels dataset.
Number of classes: 9
Data shapes:
  Xtr: (270, 29, 12)
  Ytr: (270, 1)
  Xte: (370, 29, 12)
  Yte: (370, 1)


In [ ]:
# 由于我们正在进行聚类，不需要训练/测试分割
X = np.concatenate((Xtr, Xte), axis=0)
Y = np.concatenate((Ytr, Yte), axis=0)

## 初始化和拟合 RC 模型

In [5]:
rcm =  RC_model(**config)

In [ ]:
# 生成输入 MTS 的表示
rcm.fit(X)
mts_representations = rcm.input_repr

Training completed in 0.02 min


## 计算聚类分区

In [ ]:
# 计算相异度矩阵
Dist = cosine_distances(mts_representations)
distArray = ssd.squareform(Dist)

In [ ]:
# 层次聚类
distArray = ssd.squareform(Dist)
Z = linkage(distArray, 'ward')
clust = fcluster(Z, t=4.0, criterion="distance")
print(f"Found {len(np.unique(clust))} clusters")

Found 9 clusters


In [ ]:
# 评估类别标签和聚类标签之间的一致性
nmi = v_measure_score(Y[:,0], clust)
print(f"Normalized Mutual Information (v-score): {nmi:.3f}")

Normalized Mutual Information (v-score): 0.899
